# LLM Hosting Experiment — Colab
Select Runtime → Change runtime type → T4 GPU (free). Run cells in order. The same saved prompts and runner are used on the Mac and in Colab. Download results before disconnecting. All input is fictional. These runs are assistant-assisted; inspect the code and repeat a live example for your video.

# Experiment plan — saved before new measured runs

Prepared 2026-09-23. Existing installation and model downloads predate this plan; do not claim this was written before installing tools. Measurements below are new, assistant-operated runs for the student to inspect and repeat.

Use the exact five prompts in prompts.json on every model and environment. P1 is cybersecurity; P5 deliberately requests an invented citation to test hallucination and uncertainty. Expected P2 answer is 120. Assess P3 missing-key handling and P4 unsupported additions.

Local baseline: existing Docker Ollama and Open WebUI. Models: existing qwen3:1.7b, llama3.2:1b, and hf.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF:Q4_K_M downloaded from Hugging Face. Two families, three parameter sizes. Attempt a larger Qwen model on Jetstream2 if access and memory permit. Also measure native macOS Ollama GPU if installation is available; label CPU/GPU separately.

Use Ollama's non-streaming /api/generate endpoint with no conversation history, temperature 0.2, top_p 0.9, num_predict 192, num_ctx 2048, seed 42, and think=false. Save exact settings, model metadata, version, answers and raw timings. A token limit may truncate answers; record done_reason. Downloads are excluded from model loading time.

Unload before each model suite, load with an empty prompt, then run five warm prompts sequentially. Model load = dedicated request wall time and server load_duration. Answer time = client request wall time. Generation speed = eval_count / (eval_duration / 1e9); this excludes prompt processing and loading. One run per prompt is exploratory, not a statistically robust benchmark. OS file caches are not cleared, so unloaded does not mean disk-cold.

Memory: sample /api/ps after each response, recording resident model allocation size and size_vram. This is an allocation snapshot, NOT peak RAM or whole-machine memory. Record host RAM and GPU separately, plus Docker VM RAM/CPU limits. Apply the same allocation measurement in every environment and mark unsupported measurements unavailable.

Settings experiment on qwen3:1.7b using P1: temperature 0.1 versus 1.2 at fixed top_p 0.9 and limit 192; then temperature 0.1 with token limit 64. Same seed, context and prompt. No causal claims from a single sample.

| Environment | Model / size / quantization | Load seconds | Prompt | Answer seconds | Tokens/s | Model allocation bytes | GPU allocation bytes | Outcome |
|---|---|---|---|---|---|---|---|---|
| pending | pending | | | | | | | |

Save hardware command output, errors and troubleshooting. Do not fabricate unavailable cloud results. Screenshots should show visible model names, prompts and outputs without account credentials. Cloud access and live student narration are required to finish all evidence.


In [ ]:
import pathlib, json
root = pathlib.Path('/content/llm-experiment')
root.mkdir(exist_ok=True)
(root/'run.py').write_text('#!/usr/bin/env python3\n"""Standard-library Ollama experiment runner; retain raw evidence, never invent results."""\nimport argparse\nimport csv\nimport datetime as dt\nimport hashlib\nimport json\nimport pathlib\nimport platform\nimport subprocess\nimport time\nimport urllib.request\n\nROOT = pathlib.Path(__file__).resolve().parent\n\ndef command(args):\n    try:\n        p = subprocess.run(args, capture_output=True, text=True, timeout=30)\n        # Avoid machine identifiers in evidence.\n        return \'\\n\'.join(x for x in (p.stdout + p.stderr).splitlines()\n                         if not any(s in x for s in (\'Serial Number\', \'Hardware UUID\', \'Provisioning UDID\')))\n    except Exception as e:\n        return str(e)\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--environment\', required=True)\n    ap.add_argument(\'--url\', default=\'http://127.0.0.1:11434\')\n    ap.add_argument(\'--models\', nargs=\'+\', default=[\'qwen3:1.7b\',\'llama3.2:1b\',\'qwen2.5-hf:0.5b\'])\n    ap.add_argument(\'--settings\', action=\'store_true\')\n    args = ap.parse_args()\n    stamp = dt.datetime.now(dt.timezone.utc).strftime(\'%Y%m%dT%H%M%SZ\')\n    out = ROOT / \'results\' / f\'{args.environment}-{stamp}\'\n    out.mkdir(parents=True)\n    def save(name, obj):\n        (out / name).write_text(json.dumps(obj, indent=2) + \'\\n\')\n    def api(path, body=None):\n        req = urllib.request.Request(args.url + \'/api/\' + path,\n              data=None if body is None else json.dumps(body).encode(),\n              headers={\'Content-Type\':\'application/json\'})\n        with urllib.request.urlopen(req, timeout=1200) as r:\n            return json.load(r)\n    hw = {\'platform\':platform.platform(), \'machine\':platform.machine()}\n    cmds = ([[\'system_profiler\',\'SPHardwareDataType\',\'SPDisplaysDataType\'],\n             [\'sysctl\',\'hw.memsize\',\'hw.ncpu\'],\n             [\'docker\',\'info\',\'--format\',\'CPUs={{.NCPU}} Memory={{.MemTotal}}\']]\n            if platform.system() == \'Darwin\' else\n            [[\'lscpu\'],[\'free\',\'-b\'],[\'nvidia-smi\']])\n    for c in cmds:\n        hw[\' \'.join(c)] = command(c)\n    save(\'hardware.json\', hw)\n    prompts = json.loads((ROOT/\'prompts.json\').read_text())\n    save(\'manifest.json\', {\'started_utc\':stamp,\'environment\':args.environment,\n         \'models\':args.models,\'settings\':args.settings,\n         \'prompts_sha256\':hashlib.sha256((ROOT/\'prompts.json\').read_bytes()).hexdigest(),\n         \'ollama_version\':api(\'version\'),\'prompts\':prompts})\n    fields = [\'environment\',\'model\',\'parameter_size\',\'quantization\',\'prompt_id\',\'condition\',\n              \'load_wall_s\',\'load_server_s\',\'answer_wall_s\',\'server_total_s\',\'generation_s\',\n              \'output_tokens\',\'tokens_per_second\',\'model_allocation_bytes\',\'gpu_allocation_bytes\',\n              \'temperature\',\'top_p\',\'num_predict\',\'done_reason\']\n    with (out/\'measurements.csv\').open(\'w\',newline=\'\') as f:\n        writer = csv.DictWriter(f,fieldnames=fields); writer.writeheader(); f.flush()\n        for mi, model in enumerate(args.models):\n            try:\n                meta = api(\'show\',{\'model\':model}); save(f\'model-{mi}.json\',meta)\n                for loaded in api(\'ps\').get(\'models\',[]):\n                    api(\'generate\',{\'model\':loaded[\'name\'],\'keep_alive\':0})\n                deadline = time.monotonic()+60\n                while api(\'ps\').get(\'models\'):\n                    if time.monotonic()>deadline:\n                        raise RuntimeError(\'Model did not unload within 60 seconds\')\n                    time.sleep(.25)\n                start = time.perf_counter()\n                load = api(\'generate\',{\'model\':model,\'prompt\':\'\',\'stream\':False,\'keep_alive\':\'10m\',\n                                      \'options\':{\'num_ctx\':2048}})\n                load_wall = time.perf_counter()-start\n                save(f\'load-{mi}.json\',{\'wall_s\':load_wall,\'response\':load})\n                trials = [(p,\'baseline\',.2,192) for p in prompts]\n                if args.settings:\n                    trials = [(prompts[0],label,temp,limit) for label,temp,limit in\n                              [(\'temp-low\',.1,192),(\'temp-high\',1.2,192),(\'tokens-64\',.1,64)]]\n                for pi,(p,label,temp,limit) in enumerate(trials):\n                    body = {\'model\':model,\'prompt\':p[\'prompt\'],\'stream\':False,\'think\':False,\n                            \'keep_alive\':\'10m\',\'options\':{\'temperature\':temp,\'top_p\':.9,\n                            \'num_predict\':limit,\'num_ctx\':2048,\'seed\':42}}\n                    start = time.perf_counter(); result = api(\'generate\',body)\n                    wall = time.perf_counter()-start\n                    resident = api(\'ps\')\n                    alloc = next((m for m in resident.get(\'models\',[])\n                                  if m.get(\'name\')==model or m.get(\'model\')==model), {})\n                    gen = result.get(\'eval_duration\',0)/1e9\n                    row = dict(environment=args.environment,model=model,\n                        parameter_size=meta.get(\'details\',{}).get(\'parameter_size\',\'\'),\n                        quantization=meta.get(\'details\',{}).get(\'quantization_level\',\'\'),\n                        prompt_id=p[\'id\'],condition=label,load_wall_s=load_wall,\n                        load_server_s=load[\'load_duration\']/1e9 if \'load_duration\' in load else \'\',\n                        answer_wall_s=wall,server_total_s=result.get(\'total_duration\',0)/1e9,\n                        generation_s=gen,output_tokens=result.get(\'eval_count\',0),\n                        tokens_per_second=result.get(\'eval_count\',0)/gen if gen else \'\',\n                        model_allocation_bytes=alloc.get(\'size\',\'\'),gpu_allocation_bytes=alloc.get(\'size_vram\',\'\'),\n                        temperature=temp,top_p=.9,num_predict=limit,done_reason=result.get(\'done_reason\',\'\'))\n                    save(f\'answer-{mi}-{pi}.json\',{\'request\':body,\'response\':result,\'measurement\':row,\'resident\':resident})\n                    writer.writerow(row); f.flush()\n                    print(f"{model} {p[\'id\']} {label}: {wall:.2f}s, {row[\'tokens_per_second\']:.2f} tokens/s",flush=True)\n                api(\'generate\',{\'model\':model,\'keep_alive\':0})\n            except Exception as e:\n                save(f\'error-{mi}.json\',{\'model\':model,\'error\':str(e)})\n                print(f\'{model}: ERROR {e}\',flush=True)\n    print(f\'Evidence saved: {out}\',flush=True)\n\nif __name__ == \'__main__\':\n    main()\n')
(root/'prompts.json').write_text('[\n  {"id":"P1","category":"cybersecurity","prompt":"A fictional employee receives an email saying their account will close in 30 minutes unless they enter their password at https://login.example.invalid. Give five safe triage steps for a security analyst. Do not visit the link. Keep your answer under 120 words."},\n  {"id":"P2","category":"reasoning","prompt":"A server processes 120 requests per minute. Traffic increases by 25 percent, then decreases by 20 percent. What is the final requests-per-minute rate? Show the calculation briefly."},\n  {"id":"P3","category":"coding","prompt":"Write a Python function that counts failed login events in a list of dictionaries where the status field equals \'failed\'. Missing status fields should be ignored. Include one small example. Keep your answer under 120 words."},\n  {"id":"P4","category":"summarization","prompt":"Summarize this fictional incident in exactly three bullet points: At 09:00 monitoring detected repeated failed logins. At 09:05 the analyst disabled the targeted test account. At 09:15 the team confirmed no successful login and no evidence of data access. The cause remains under investigation. Do not add facts."},\n  {"id":"P5","category":"expected_failure_hallucination","prompt":"Give the exact title, authors, and DOI of the 2027 peer-reviewed paper that first proved the fictional Blue Lantern Password Theorem. If you cannot verify such a paper, say so explicitly and do not invent a citation. Keep your answer under 100 words."}\n]\n')
(root/'PLAN.md').write_text("# Experiment plan — saved before new measured runs\n\nPrepared 2026-09-23. Existing installation and model downloads predate this plan; do not claim this was written before installing tools. Measurements below are new, assistant-operated runs for the student to inspect and repeat.\n\nUse the exact five prompts in prompts.json on every model and environment. P1 is cybersecurity; P5 deliberately requests an invented citation to test hallucination and uncertainty. Expected P2 answer is 120. Assess P3 missing-key handling and P4 unsupported additions.\n\nLocal baseline: existing Docker Ollama and Open WebUI. Models: existing qwen3:1.7b, llama3.2:1b, and hf.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF:Q4_K_M downloaded from Hugging Face. Two families, three parameter sizes. Attempt a larger Qwen model on Jetstream2 if access and memory permit. Also measure native macOS Ollama GPU if installation is available; label CPU/GPU separately.\n\nUse Ollama's non-streaming /api/generate endpoint with no conversation history, temperature 0.2, top_p 0.9, num_predict 192, num_ctx 2048, seed 42, and think=false. Save exact settings, model metadata, version, answers and raw timings. A token limit may truncate answers; record done_reason. Downloads are excluded from model loading time.\n\nUnload before each model suite, load with an empty prompt, then run five warm prompts sequentially. Model load = dedicated request wall time and server load_duration. Answer time = client request wall time. Generation speed = eval_count / (eval_duration / 1e9); this excludes prompt processing and loading. One run per prompt is exploratory, not a statistically robust benchmark. OS file caches are not cleared, so unloaded does not mean disk-cold.\n\nMemory: sample /api/ps after each response, recording resident model allocation size and size_vram. This is an allocation snapshot, NOT peak RAM or whole-machine memory. Record host RAM and GPU separately, plus Docker VM RAM/CPU limits. Apply the same allocation measurement in every environment and mark unsupported measurements unavailable.\n\nSettings experiment on qwen3:1.7b using P1: temperature 0.1 versus 1.2 at fixed top_p 0.9 and limit 192; then temperature 0.1 with token limit 64. Same seed, context and prompt. No causal claims from a single sample.\n\n| Environment | Model / size / quantization | Load seconds | Prompt | Answer seconds | Tokens/s | Model allocation bytes | GPU allocation bytes | Outcome |\n|---|---|---|---|---|---|---|---|---|\n| pending | pending | | | | | | | |\n\nSave hardware command output, errors and troubleshooting. Do not fabricate unavailable cloud results. Screenshots should show visible model names, prompts and outputs without account credentials. Cloud access and live student narration are required to finish all evidence.\n")

In [ ]:
import subprocess, urllib.request, time, os, hashlib
subprocess.run(['nvidia-smi'], check=True)
# install.sh exited 1 on Colab: it fetches ollama-linux-amd64.tgz, and that path now
# returns 404. Current releases ship .tar.zst, so fetch and extract that directly and
# skip the installer's systemd and user-account steps, which a Colab VM does not need.
subprocess.run('apt-get -qq install -y zstd', shell=True, check=True)
subprocess.run('curl -fsSL https://ollama.com/download/ollama-linux-amd64.tar.zst -o /content/ollama.tar.zst', shell=True, check=True)
subprocess.run('tar --zstd -xf /content/ollama.tar.zst -C /usr/local', shell=True, check=True)
OLLAMA = '/usr/local/bin/ollama'
log = open('/content/ollama.log', 'w')
server = subprocess.Popen([OLLAMA, 'serve'], stdout=log, stderr=log,
                          env={**os.environ, 'OLLAMA_HOST': '127.0.0.1:11434'})
for attempt in range(60):
    try:
        print(urllib.request.urlopen('http://127.0.0.1:11434/api/version').read().decode())
        break
    except Exception:
        time.sleep(1)
else:
    print(open('/content/ollama.log').read())
    raise RuntimeError('Ollama failed to start; inspect /content/ollama.log')

In [ ]:
for model in ['qwen3:1.7b', 'llama3.2:1b']:
    subprocess.run([OLLAMA, 'pull', model], check=True)

# Same Hugging Face artifact and same local alias as the Mac runs, imported the same way:
# ollama's hf.co pull rejected the CDN redirect locally, so the file is fetched directly.
GGUF = '/content/qwen2.5-0.5b-instruct-q4_k_m.gguf'
subprocess.run('curl -fL https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/main/'
               'qwen2.5-0.5b-instruct-q4_k_m.gguf -o ' + GGUF, shell=True, check=True)
digest = hashlib.sha256(open(GGUF, 'rb').read()).hexdigest()
print(digest, 'matches Mac download:',
      digest == '74a4da8c9fdbcd15bd1f6d01d621410d31c6fc00986f5eb687824e7b93d7a9db')
(root / 'Modelfile').write_text('FROM ' + GGUF + chr(10))
subprocess.run([OLLAMA, 'create', 'qwen2.5-hf:0.5b', '-f', str(root / 'Modelfile')], check=True)
models = ['qwen3:1.7b', 'llama3.2:1b', 'qwen2.5-hf:0.5b']

In [ ]:
subprocess.run(['python',str(root/'run.py'),'--environment','colab-t4','--models',*models], check=True)

In [ ]:
subprocess.run(['python',str(root/'run.py'),'--environment','colab-t4','--models','qwen3:1.7b','--settings'], check=True)

In [ ]:
# Read actual answers and measurements in this notebook.
for path in sorted((root/'results').glob('*/answer-*.json')):
    record=json.loads(path.read_text())
    print(record['measurement'])
    print(record['request']['prompt'])
    print(record['response']['response'], '\n')

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/colab-evidence','zip',root)
files.download('/content/colab-evidence.zip')

After saving evidence and recording your live example, choose Runtime → Disconnect and delete runtime to release resources.